In [ ]:
import requests
import time
import pandas as pd
from datetime import datetime

# T.I.D.E. Engine
### Trending Index Dynamic Execution

## Overview
T.I.D.E. is an automated Python trading system engineered for live market execution. It runs monthly rotation checks across a 25-asset equities basket, calculating trailing 6-month momentum (127 trading days) via TwelveData API and executing market orders via Alpaca paper-trading endpoints.

The core strategy logic was rigorously validated across three historical periods before being connected to active broker APIs.

## Key Features & Architecture
- **Trailing Momentum Engine:** Pulls daily time-series market data to evaluate 6-month performance across 25 large-cap stocks (Tech, Healthcare, Finance, Energy, Consumer).
- **Risk Protection (Cash Fallback):** Automatically shifts target position to 100% CASH if top asset momentum falls to 0% or below.
- **Automated Order Pipeline:** Manages position rebalancing, market order placement, and position tracking via Alpaca broker endpoints.
- **Account & Event Logging:** Features a built-in `TradingAccount` class that tracks portfolio balance, logs P&L adjustments, and records trading events to local CSV logs.

## Tech Stack
- **Language:** Python 3
- **Data & APIs:** TwelveData API (Time Series), Alpaca REST API (Trading & Data)
- **Libraries:** Pandas, Requests, Datetime, Time

ALPACA_KEY_ID = "PASTE_YOUR_ALPACA_KEY_ID_HERE"
ALPACA_SECRET_KEY = "PASTE_YOUR_ALPACA_SECRET_HERE"
ALPACA_BASE_URL = "https://paper-api.alpaca.markets"

TWELVE_DATA_API_KEY = "PASTE_YOUR_TWELVE_DATA_KEY_HERE"

STARTING_BALANCE = 100
PAYOUT_THRESHOLD = 8000
PAYOUT_AMOUNT = 100

HEADERS = {
    "APCA-API-KEY-ID": ALPACA_KEY_ID,
    "APCA-API-SECRET-KEY": ALPACA_SECRET_KEY,
}

TICKERS = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "ADBE", "CRM", "CSCO", "NFLX",
    "JNJ", "UNH", "MRK", "PFE",
    "JPM", "V", "MA", "BAC",
    "PG", "KO", "PEP", "WMT",
    "HD", "DIS",
    "XOM",
    "T", "VZ",
]


# -------------------------
# ALPACA FUNCTIONS
# --------------------------

def get_account():
    resp = requests.get(f"{ALPACA_BASE_URL}/v2/account", headers=HEADERS)
    resp.raise_for_status()
    return resp.json()

def get_latest_price(symbol):
    url = f"https://data.alpaca.markets/v2/stocks/{symbol}/trades/latest"
    resp = requests.get(url, headers=HEADERS)
    resp.raise_for_status()
    return float(resp.json()["trade"]["p"])

def place_order(symbol, qty, side):
    order = {
        "symbol": symbol,
        "qty": qty,
        "side": side,
        "type": "market",
        "time_in_force": "day",
    }
    resp = requests.post(f"{ALPACA_BASE_URL}/v2/orders", headers=HEADERS, json=order)
    resp.raise_for_status()
    return resp.json()

def get_open_position(symbol):
    resp = requests.get(f"{ALPACA_BASE_URL}/v2/positions/{symbol}", headers=HEADERS)
    if resp.status_code == 404:
        return None
    resp.raise_for_status()
    return resp.json()


# ---------------------
# ACCOUNT TRACKING
# ---------------------

class TradingAccount:
    def __init__(self, starting_balance, payout_threshold, payout_amount, log_path="tide_model_two_log.csv"):
        self.balance = starting_balance
        self.principal = starting_balance
        self.payout_threshold = payout_threshold
        self.payout_amount = payout_amount
        self.log_path = log_path
        self.history = []
        self._log_event("account_opened", self.balance)

    def apply_trade_result(self, pnl_dollars):
        self.balance += pnl_dollars
        self._log_event("month_closed", self.balance, note=f"pnl {round(pnl_dollars, 2)}")
        self._check_payout_eligibility()

    def add_deposit(self, amount):
        self.balance += amount
        self.principal += amount
        self.payout_threshold += amount
        self._log_event("manual_deposit", self.balance, note=f"deposit {amount}")

    def _check_payout_eligibility(self):
        if self.balance >= self.payout_threshold:
            payout = min(self.payout_amount, self.balance - self.principal)
            if payout > 0:
                self.balance -= payout
                self._log_event("payout_released", self.balance, note=f"paid out {round(payout, 2)}")

    def _log_event(self, event_type, balance, note=""):
        entry = {
            "timestamp": datetime.now().isoformat(),
            "event": event_type,
            "balance": round(balance, 2),
            "principal": round(self.principal, 2),
            "note": note,
        }
        self.history.append(entry)
        pd.DataFrame(self.history).to_csv(self.log_path, index=False)

    def status(self):
        return {
            "current_balance": round(self.balance, 2),
            "principal_protected": round(self.principal, 2),
            "profit_above_principal": round(self.balance - self.principal, 2),
            "next_payout_threshold": round(self.payout_threshold, 2),
        }


# ----------------------------------------------
# ROTATION CHECK SAME MOMENTUM AS BACKTEST
# ----------------------------------------------

def get_six_month_momentum(symbol):
    """Pulls daily data and returns the trailing six month price return for one ticker."""
    url = "https://api.twelvedata.com/time_series"
    params = {
        "symbol": symbol,
        "interval": "1day",
        "outputsize": 200,
        "apikey": TWELVE_DATA_API_KEY,
        "format": "JSON",
    }
    resp = requests.get(url, params=params, timeout=15)
    data = resp.json()
    if "values" not in data:
        print(f"Could not fetch data for {symbol}, {data.get('message', data)}")
        return None

    df = pd.DataFrame(data["values"])
    df["datetime"] = pd.to_datetime(df["datetime"])
    df["close"] = pd.to_numeric(df["close"])
    df = df.sort_values("datetime").reset_index(drop=True)

    if len(df) < 127:
        return None

    six_months_ago_price = df["close"].iloc[-127]
    current_price = df["close"].iloc[-1]
    momentum = (current_price / six_months_ago_price) - 1
    return momentum


def run_monthly_rotation_check(account, current_holding):
    """
    Checks momentum across the whole basket, decides the best holder or cash,
    and if that differs from what is currently held, sells the old position
    and buys the new one through Alpaca, logging the result either way.
    Call this once per month.
    """
    print("Checking momentum across the basket...")
    momentum_scores = {}
    for t in TICKERS:
        score = get_six_month_momentum(t)
        if score is not None:
            momentum_scores[t] = score
        time.sleep(8)  # pace requests under the free tier limit

    if not momentum_scores:
        print("Could not retrieve momentum for any ticker, skipping this cycle.")
        return current_holding

    best_ticker = max(momentum_scores, key=momentum_scores.get)
    best_score = momentum_scores[best_ticker]
    target_holding = "CASH" if best_score <= 0 else best_ticker

    print(f"Best ticker, {best_ticker}, six month momentum {round(best_score * 100, 2)}%")
    print(f"Target holding for this month, {target_holding}")

    if target_holding == current_holding:
        print("No change needed, already holding the correct position.")
        return current_holding

    # Sell whatever is currently held, if anything
    if current_holding != "CASH":
        position = get_open_position(current_holding)
        if position:
            qty = position["qty"]
            entry_price = float(position["avg_entry_price"])
            place_order(current_holding, qty, "sell")
            exit_price = get_latest_price(current_holding)
            pnl_pct = (exit_price - entry_price) / entry_price * 100
            pnl_dollars = account.balance * (pnl_pct / 100)
            account.apply_trade_result(pnl_dollars)
            print(f"Sold {current_holding}, result {round(pnl_pct, 2)}%")

    # Buy the new target, if it is not cash
    if target_holding != "CASH":
        price = get_latest_price(target_holding)
        qty = round(account.balance / price, 4)
        place_order(target_holding, qty, "buy")
        print(f"Bought {target_holding} at {price}")

    return target_holding


# ---------
# RUN IT
# ---------

account = TradingAccount(starting_balance=STARTING_BALANCE, payout_threshold=PAYOUT_THRESHOLD, payout_amount=PAYOUT_AMOUNT)
current_holding = "CASH"  # update this manually if resuming with an existing position already held

print("Alpaca account status.")
print(get_account())

current_holding = run_monthly_rotation_check(account, current_holding)
